# Week 6 -- Function 6: Bayesian Optimisation (5D)

In [ ]:
# Cell 2: Imports
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

In [ ]:
# Cell 3: Load data and inspect
# Function 6: Cake recipe optimisation (5D), Maximise.
# Note: scores are negative by design -- maximising means bringing the score closest to zero.

X = np.load('../data/function_6/initial_inputs.npy')
Y = np.load('../data/function_6/initial_outputs.npy')

print(f'Input  shape : {X.shape}   (n_samples x n_dimensions)')
print(f'Output shape : {Y.shape}  (n_samples,)')
print()
print('  Note: Y scores are negative by design.')
print('  Maximising = finding the recipe closest to 0 (least penalised).')
print()

# Sort descending by Y value
X_list = list(X)
Y_list = list(Y)
pairs = sorted(zip(Y_list, X_list), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 86)
print('  All observations (sorted descending by Y)')
print('=' * 86)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    print(f'  [{i+1:2d}]  X = [{x_val[0]:.6f}, {x_val[1]:.6f}, {x_val[2]:.6f}, {x_val[3]:.6f}, {x_val[4]:.6f}]   Y = {y_val:+.6f}{marker}')
print('=' * 86)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
print(f'\n  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{best_X[0]:.8f}, {best_X[1]:.8f}, {best_X[2]:.8f}, {best_X[3]:.8f}, {best_X[4]:.8f}]')

In [ ]:
# Cell 4: Fit GP with StandardScaler-transformed Y
# Y is always negative by design -- log(|Y|) would reverse ordering (more negative = larger |Y| = higher log).
# StandardScaler centres and scales Y, preserving the correct ordering for UCB maximisation.

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
Y_fit = scaler.fit_transform(Y.reshape(-1, 1)).ravel()
# StandardScaler used -- Y is always negative by design, log(|Y|) would reverse the ordering

# Fixed RBF kernel (course style -- no ConstantKernel, no optimisation)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                   : {gp.kernel_}')
print(f'  Log-marginal-likelihood  : {gp.log_marginal_likelihood_value_:.4f}')
print()

# Sanity check: predict at best known point
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_scaled = scaler.transform([[best_Y]])[0][0]
print('  Sanity check at best known X*:')
print(f'    X*                     = [{best_X[0]:.6f}, {best_X[1]:.6f}, {best_X[2]:.6f}, {best_X[3]:.6f}, {best_X[4]:.6f}]')
print(f'    GP predicted mean      = {pred_mean[0]:.4f}  (scaled-space)')
print(f'    Actual scaled Y*       = {actual_scaled:.4f}  (scaled-space)')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
print('=' * 62)

In [ ]:
# Cell 5: UCB acquisition over random search (20,000 points in 5D)

np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(20000, 5))  # shape (20000, 5)

# GP predictions across the random grid
post_mean, post_std = gp.predict(X_grid, return_std=True)

# UCB acquisition: mean + beta * std
beta = 2.5  #Higher beta = more exploration at early stage (Week 1, sparse data)
acquisition = post_mean + beta * post_std  # shape (20000,)

# Next query = grid point with highest UCB
best_idx = np.argmax(acquisition)
next_x   = X_grid[best_idx]
next_acq = acquisition[best_idx]

# Portal submission string
portal_string = f'{next_x[0]:.6f}-{next_x[1]:.6f}-{next_x[2]:.6f}-{next_x[3]:.6f}-{next_x[4]:.6f}'

print('=' * 62)
print('  UCB Acquisition  (beta = 2.5, random search 20,000 pts)')
print('=' * 62)
print(f'  Grid size            : {len(X_grid)} points (random uniform, 5D)')
print(f'  Max UCB value        : {next_acq:.4f}')
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}, {next_x[2]:.6f}, {next_x[3]:.6f}, {next_x[4]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

In [ ]:
# Cell 7: Summary

print('=' * 62)
print('  SUMMARY -- Bayesian Optimisation Results')
print('=' * 62)
print(f'  Function             : 5D Cake Recipe Optimisation')
print(f'  Objective            : Maximise (scores negative by design; closer to 0 = better)')
print(f'  Kernel               : RBF(length_scale=0.1, fixed)')
print(f'  Acquisition function : UCB  (beta = 2.5)')
print(f'  Y transform          : StandardScaler (Y always negative)')
print(f'  Grid search          : Random uniform (20,000 points, 5D)')
print()
print(f'  Current best X*      : [{best_X[0]:.6f}, {best_X[1]:.6f}, {best_X[2]:.6f}, {best_X[3]:.6f}, {best_X[4]:.6f}]')
print(f'  Current best Y*      : {best_Y:.6f}')
print()
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}, {next_x[2]:.6f}, {next_x[3]:.6f}, {next_x[4]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

## Week 1 -- Reflection

**Function**: F6 -- Cake Recipe Scoring (5D)  
**Domain context**: Score is negative by design; goal is to bring it as close to 0 as possible

### Query
- **Method**: GP + UCB (beta=2.5), RBF kernel (length_scale=0.1, fixed), Random 20,000 pts (5D)
- **Query type**: Exploration -- first submission, high uncertainty everywhere
- **Input submitted**: [0.705342, 0.105077, 0.763664, 0.783759, 0.051342]
- **Output received**: -0.695846

### What this result taught us
Score y=-0.696 (negative by design). High x1/x3/x4 with low x2/x5 gave a moderate result; other recipes may score better.

### Strategy for Week 2
Try markedly different recipe -- lower x1/x3/x4, raise x2 slightly; the 5D space may have better combinations far from this point.